In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr
import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances

from sklearn.metrics import silhouette_score
import scipy.stats
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering

import scanpy as sc
import anndata
import scipy as sp
import scipy.sparse
import dask.array as da
# from igraph.drawing.text import TextDrawer
from tslearn.neighbors import KNeighborsTimeSeries
from tslearn.metrics import cdist_soft_dtw_normalized,cdist_soft_dtw
import networkx as nx
import igraph as ig
import leidenalg
import umap
from scanpy.plotting.palettes import default_20,vega_20_scanpy
from matplotlib import pyplot as plt
from matplotlib import cm
import ast
from tslearn.barycenters import softdtw_barycenter,dtw_barycenter_averaging,euclidean_barycenter
import xarray as xr

import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import re
import goatools
import goatools.base
from goatools.base import download_go_basic_obo

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.semantic import semantic_similarity
from goatools.semantic import TermCounts, get_info_content

from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.go_enrichment import GOEnrichmentStudy

import holoviews as hv
hv.extension('bokeh')
random.seed(42)
np.random.seed(42)

warnings.filterwarnings(action='once',category=UserWarning)

from IPython.display import Video

In [ ]:
def highlight_regex(an_df,regex,column_label="Gene",color_all_mode="red",highlight="all",pallete=vega_20_scanpy,match_mode="regex",fontsize=18):
    ## highlight ("all","each") will either render all matches as a single color ("all") or each match type (e.g. gene name) as its own color ("each") acoording to the pallete
    ## match_mode ("regex","exact") will either perform a re.search ("regex") or exact match ("exact") for its search
    ## regex may be either (1) a regex string, (2) an exact match string or (3) a list (when in exact match mode) of exact matches to search
    
    
    highlight_df = copy.deepcopy(an_df)
    regex_title = regex
    
    if match_mode == "regex":
        re_pattern = re.compile(regex)
        regex_mask = highlight_df.obs[column_label].apply(lambda x: re.search(re_pattern,x) != None)
        regex_name_list = sorted(highlight_df.obs[regex_mask][column_label].unique().tolist())
    
    elif match_mode == "exact":
        if type(regex) == list:
            regex_mask = highlight_df.obs[column_label].apply(lambda x: any([x==regex_val for regex_val in regex]))
            regex_title = ",".join(regex_title)
        else:
            regex_mask = highlight_df.obs[column_label].apply(lambda x: x==regex)
        regex_name_list = sorted(highlight_df.obs[regex_mask][column_label].unique().tolist())
    
    if highlight == "all":
        highlight_df.obs[column_label + " | Regex: " + regex_title] = regex_mask.astype("category")
        fig = sc.pl.umap(highlight_df, title=column_label + " | Regex: " + regex_title, color=column_label + " | Regex: " + regex_title,
                     groups=[True], show=False, legend_loc='right margin', add_outline=False, size=50, return_fig=True,
                     palette = {True:color_all_mode,False:"lightgrey"});
    elif highlight == "each":
        highlight_df.obs[column_label + " | Regex: " + regex_title] = highlight_df.obs[column_label].apply(lambda x: x if x in regex_name_list else np.NaN).astype("category")
        fig = sc.pl.umap(highlight_df, title=column_label + " | Regex: " + regex_title, color=column_label + " | Regex: " + regex_title, groups=regex_name_list, show=False, legend_loc='right margin', \
                         add_outline=False, size=50, return_fig=True,palette=pallete);
    
    
    axes = fig.get_axes()
    for ax in axes:
        ax.set_title(ax.get_title(), fontsize=fontsize)
        ax.set_ylabel(ax.get_ylabel(), fontsize=fontsize)
        ax.set_xlabel(ax.get_xlabel(), fontsize=fontsize)
        
    return fig
    
def highlight_cluster(an_df,cluster_level,cluster_label,resolution_list,color="purple",fontsize=18):
    
    fig = highlight_regex(an_df,cluster_label,column_label=cluster_level,color_all_mode=color,highlight="all",match_mode="exact",fontsize=fontsize);
    
    cluster_level_index = int(cluster_level[1:])
    
    axes = fig.get_axes()
    for i,ax in enumerate(axes):
        if i%2==0:
            ax.set_title("Leiden Resolution=" + str(resolution_list[cluster_level_index]) + " | " + str(cluster_label), fontsize=fontsize)
            
    return fig

def search_go(ns2assoc, obodag, inv_gene_to_id, go_term):
    namespace_abbv = {"biological_process":"BP","molecular_function":"MF","cellular_component":"CC"}
    
    print("Searching for " + str(obodag[go_term].name))
    namespace = namespace_abbv[obodag[go_term].namespace]
    child_goterms = list(obodag[go_term].get_all_children())
    gene_list = [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if go_term in val]
    for child_goterm in child_goterms:    
        gene_list += [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if child_goterm in val]
    gene_list = sorted(list(set(gene_list)))
    return gene_list

def highlight_go(an_df, ns2assoc, obodag, inv_gene_to_id, go_term, go_mode = "all", color="red", fontsize=18):
    
    gene_list = search_go(ns2assoc, obodag, inv_gene_to_id, go_term)
    if go_mode == "all":
        fig = highlight_regex(an_df,gene_list,column_label="Gene",color_all_mode=color,highlight="all",match_mode="exact",fontsize=fontsize)
    elif go_mode == "each":
        fig = highlight_regex(an_df,gene_list,column_label="Gene",highlight="each",match_mode="exact",fontsize=fontsize)
        
    plt.title(go_term)
    return fig

def get_enriched_GO_terms(background_gene_list,gene_list,obodag,objanno,ns2assoc,pval=0.05,GO_type="BP"):
    
    gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
    synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
    gene_to_id.update(synonym_dict)

    #background gene set

    all_genes_uniprot = [gene_to_id[item] for item in background_gene_list if item in gene_to_id.keys()]
    selected_genes_uniprot = [gene_to_id[item] for item in gene_list if item in gene_to_id.keys()]
    
    print(len(all_genes_uniprot))
    print(len(selected_genes_uniprot))
    
    goeaobj = GOEnrichmentStudy(
            all_genes_uniprot, # List of mouse protein-coding genes
            ns2assoc[GO_type], # geneid/GO associations
            obodag, # Ontologies
            propagate_counts = True,
            alpha = pval, # default significance cut-off
            methods=["fdr_bh"]); # defult multipletest correction method

    goea_results_all = goeaobj.run_study(selected_genes_uniprot, prt=None)
    goea_quiet_sig = [r for r in goea_results_all if r.p_fdr_bh < pval]
    goea_quiet_enriched = [r for r in goea_quiet_sig if r.enrichment == "e"]
    return goea_quiet_enriched

def pick_exemplar(go1,go2,termcounts,obodag,info_thr,pval_factor=2.):
    
    info_1_low = get_info_content(go1.GO, termcounts)<info_thr
    info_2_low = get_info_content(go2.GO, termcounts)<info_thr
    if info_1_low and not info_2_low:
        return go2
    elif info_2_low and not info_1_low:
        return go1
    elif info_2_low and info_1_low:
        return go1
    
    pval_ratio = go1.p_fdr_bh/go2.p_fdr_bh
    
    if pval_ratio > pval_factor:
        return go2
    elif pval_ratio < (1./pval_factor):
        return go1
    
    go1_parents = list(obodag[go1.GO].get_all_parents())
    go2_parents = list(obodag[go2.GO].get_all_parents())
    
    if go2.GO in go1_parents:
        return go2
    
    elif go1.GO in go2_parents:
        return go1
    
    return go1

def get_filtered_go_terms(obodag,objanno,goea_list,sim_thr = 0.05,info_thr = 1.,GO_type="BP"):
    
    termcounts = TermCounts(obodag, objanno.get_ns2assc()[GO_type])

    go_term_list = [item.GO for item in goea_list]
    sim_arr = np.zeros((len(go_term_list),len(go_term_list)))
    for i in range(len(go_term_list)):
        for j in range(len(go_term_list)):
            sim_arr[i,j] = semantic_similarity(go_term_list[i], go_term_list[j], obodag)
    np.fill_diagonal(sim_arr, 0.)
    
    working_group_idx = 0
    grouped_terms = {}
    group_exemplars = {}
    go_term_indices = list(range(len(go_term_list)))

    while len(go_term_indices)>0:
        i = go_term_indices[0]
        most_sim_arg = np.argmax(sim_arr[i])
        sim_score = sim_arr[i,most_sim_arg]
        if sim_score > sim_thr:
            if len(grouped_terms)>0:
                in_other_group_keys = [key for key,val in grouped_terms.items() if most_sim_arg in val]
                if len(in_other_group_keys)==1:
                    other_group_idx = in_other_group_keys[0]
                    grouped_terms[other_group_idx] = grouped_terms[other_group_idx] + [i]
                    group_exemplars[other_group_idx] = pick_exemplar(group_exemplars[other_group_idx],goea_list[i],termcounts,obodag,info_thr)
                else:        
                    grouped_terms[working_group_idx] = [i,most_sim_arg]
                    group_exemplars[working_group_idx] = pick_exemplar(goea_list[i],goea_list[most_sim_arg],termcounts,obodag,info_thr)
                    working_group_idx += 1
                    go_term_indices.remove(most_sim_arg)
            else:
                grouped_terms[working_group_idx] = [i,most_sim_arg]
                group_exemplars[working_group_idx] = pick_exemplar(goea_list[i],goea_list[most_sim_arg],termcounts,obodag,info_thr)
                working_group_idx += 1
                go_term_indices.remove(most_sim_arg)
        go_term_indices.remove(i)
    
    group_exemplars = list(group_exemplars.values())
    
    return group_exemplars


def get_GO_assign_dict(selected_goea,cluster_genes_uniprot):
    all_study_items = copy.copy(cluster_genes_uniprot)
    depth_list = sorted(set([item.depth for item in selected_goea]))[::-1]
    assign_dict = {}
    for depth in depth_list:
        go_terms_at_level = [item for item in selected_goea if item.depth == depth]
        for go_term in go_terms_at_level:
            study_item_list = list(go_term.study_items)
            for study_item in study_item_list:
                if study_item in all_study_items:
                    assign_dict[study_item] = go_term.name
                    all_study_items.remove(study_item)

    for remaining_item in all_study_items:
        assign_dict[remaining_item] = "Unassigned"
        
    return assign_dict

In [ ]:
long_expt_rootpath = '/home/lag36/gutiloluis/2023-03-23_lDE26-Run1'

In [ ]:
# Get ontologies
obo_fname = download_go_basic_obo()

# Get ecoli association file (ecocyc)
gaf_handle = goatools.base.http_get("http://current.geneontology.org/annotations/ecocyc.gaf.gz",fout=long_expt_rootpath + "/ecocyc.gaf.gz")
gaf_fname = goatools.base.gunzip(long_expt_rootpath + "/ecocyc.gaf.gz")

## Getting ontologies and other nonesense

obodag = GODag(obo_fname)
objanno = GafReader(gaf_fname)
ns2assoc = objanno.get_ns2assc()

gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
inv_gene_to_id = {assoc.DB_ID:assoc.DB_Symbol for assoc in objanno.associations}
synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
gene_to_id.update(synonym_dict)

In [ ]:
def get_subset_df(df, GO_id="GO:0006633"):
    gene_names = search_go(ns2assoc, obodag, inv_gene_to_id, GO_id)
    return gene_names, df[df["Gene"].isin(gene_names)]

## TODO
- Identify and load the kernel trace file
- Make static plots of the kernel trace files at different timepoints
    - Which one was used for the L vs. growth rate plot?
- Paint them according to tRNA and ribosome genes
- Plot from both libraries
- Check the nucleoid behavior on videos/kymos of all corners of the plot. Look for the following:
    - Do nucleoid defects precede filamentation? I can also use the traces to try to answer this.
    - Does it look like a seggregation defect leading to filamentation via nucleoid occlusion?

In [ ]:
"/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-25_sgRNA_Timeseries_df.pkl"
"/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_sgRNA_Timeseries_df.pkl"

In [ ]:
df26 = pd.read_pickle("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-25_sgRNA_Timeseries_df.pkl")
df28 = pd.read_pickle("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_sgRNA_Timeseries_df.pkl")

In [ ]:
df26[df26['Gene'] == 'pssA']['Gene'].unique()

In [ ]:
cols = list(df26.columns)
[col for col in cols if 'Kernel' in col and 'z score' not in col and 'SEM' not in col]

In [ ]:
def get_two_timepoints_two_vars(df, col_x, col_y, ti, tf):

    xf = df[col_x].str[tf]
    yf = df[col_y].str[tf]
    if ti == None:
        return xf, yf
    else:
        xi = df[col_x].str[ti]
        yi = df[col_y].str[ti]
   
        return xf, yf, xi, yi

In [ ]:
trna_synthetases_GO = "GO:0006418"
fatty_acid_biosyn_GO = "GO:0006633"
trna_synthetases = search_go(ns2assoc, obodag, inv_gene_to_id, trna_synthetases_GO)

In [ ]:
xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                             col_y='Kernel Trace: major_axis_length',
                                             ti=0, tf=19)

plt.scatter(xf, yf, s=3)
plt.scatter(xi, yi, s=3)

xf, yf, xi, yi = get_two_timepoints_two_vars(df=df28,
                                             col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                             col_y='Kernel Trace: major_axis_length',
                                             ti=0, tf=19)

plt.scatter(xf, yf, s=3)
plt.scatter(xi, yi, s=3)

_, df_sub = get_subset_df(df=df26, GO_id=trna_synthetases_GO)
xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                      col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                      col_y='Kernel Trace: major_axis_length',
                                      ti=None, tf=19)
plt.scatter(xf, yf, s=5)

_, df_sub = get_subset_df(df=df28, GO_id=trna_synthetases_GO)
xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                      col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                      col_y='Kernel Trace: major_axis_length',
                                      ti=None, tf=19)
plt.scatter(xf, yf, s=5)

In [ ]:
col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
col_y='Kernel Trace: major_axis_length'

In [ ]:
df26[df26[col_y].str[19]>12]['Gene']

In [ ]:
df26[df26[col_y].str[19]<2.5]['Gene']

In [ ]:
df26

In [ ]:
list(df26.columns)

In [ ]:
df26[df26['Gene'].isin(['pssA', 'pgsA'])][
    ['Barcode',
     'sgRNA',
     'Closest Hamming Distance',
     'EcoWG1_id',
     'Gene',
     'N Mismatch',
     'Category',
     'TargetID',
     'barcodeid',
     'N Observations']]

In [ ]:
df26[df26['Gene']=='pssA']['N Mismatch']

In [ ]:
oDE_id = 2060
print(df26.loc[oDE_id]['Gene'])
Video("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/Merged/"+str(oDE_id)+".mp4", embed=True)

### Load from big library

In [ ]:
df20 = pd.read_pickle('/home/lag36/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-23_sgRNA_Timeseries_df.pkl')

In [ ]:
df20[df20['Gene']=='ftsN'][['N Mismatch', 'N Observations']]

In [ ]:
oDE_id = 24205
print(df20.loc[oDE_id]['Gene'])
Video('/home/lag36/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/MP4_Library/mCherry/'+str(oDE_id)+".mp4", embed=True)

# Observations:
- Nucleoid occlusion does not seem to be what's leading to filamentation. When cells filament in rplK KD their nucleoids are very spaced and fairly well segregated
- The nucleoid defect (compaction + increased spacing) seems to be occuring at the same time as filamentation
- Transcription defects lead to decondensed nucleoids, whereas translation defects lead to compacted nucleoids

In [ ]:
col_x, col_y

In [ ]:
df26[df26['Gene']=='ftsN']

In [ ]:
col_name = 'Kernel Trace: Total Nucleoid Area per Cell Area'
plt.plot(df26.loc[379, col_name], label = 'rplK')
plt.plot(df26.loc[27394, col_name], label = 'ftsA')
plt.plot(df26.loc[24197, col_name], label = 'ftsN')
plt.legend()
plt.ylabel(col_name)

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
col_name = 'Kernel Trace: major_axis_length'
ax1.set_xlabel('time (s)')
ax1.set_ylabel(col_name, color=color)
ax1.plot(df26.loc[379, col_name], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:blue'
col_name = 'Kernel Trace: Total Nucleoid Area per Cell Area'
ax2.set_ylabel(col_name, color=color)  # we already handled the x-label with ax1
ax2.plot(df26.loc[379, col_name], color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()

In [ ]:
cols = list(df26.columns)
[col for col in cols if 'Kernel' in col and 'z score' not in col and 'SEM' not in col]

## Secretion

In [ ]:
df26[df26['Gene']=='ffs']['N Mismatch']

In [ ]:
col_name1 = 'Kernel Trace: Nucleoid Minor Axis Length Mean'
col_name2 = 'Kernel Trace: minor_axis_length'
# plt.plot(df26.loc[7712, col_name1], label = 'ffh')
ode_id = 7712
y = df26.loc[ode_id, col_name1]/df26.loc[ode_id, col_name2]
plt.plot(y, label = 'ffh')

ode_id = 27394
y = df26.loc[ode_id, col_name1]/df26.loc[ode_id, col_name2]
plt.plot(y, label = 'ftsA')


ode_id = 21472
y = df26.loc[ode_id, col_name1]/df26.loc[ode_id, col_name2]
plt.plot(y, label = 'ffs')
plt.legend()
plt.ylabel('Nucleoid / Cell width')

In [ ]:
col_name1 = 'Kernel Trace: Nucleoid Minor Axis Length Mean'
col_name2 = 'Kernel Trace: minor_axis_length'
# plt.plot(df26.loc[7712, col_name1], label = 'ffh')
ode_id = 7712
y = df26.loc[ode_id, col_name1]
plt.plot(y, label = 'ffh')

ode_id = 27394
y = df26.loc[ode_id, col_name1]
plt.plot(y, label = 'ftsA')


ode_id = 21472
y = df26.loc[ode_id, col_name1]
plt.plot(y, label = 'ffs')
plt.legend()
plt.ylabel(col_name1)

In [ ]:
col_name1 = 'Kernel Trace: minor_axis_length'
# col_name2 = 
# plt.plot(df26.loc[7712, col_name1], label = 'ffh')
ode_id = 7712
y = df26.loc[ode_id, col_name1]
plt.plot(y, label = 'ffh')

ode_id = 27394
y = df26.loc[ode_id, col_name1]
plt.plot(y, label = 'ftsA')


ode_id = 21472
y = df26.loc[ode_id, col_name1]
plt.plot(y, label = 'ffs')
plt.legend()
plt.ylabel(col_name1)

# Bivariate plots with Kernel traces

In [ ]:
xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                             col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
                                             ti=0, tf=19)

plt.scatter(xf, yf, s=3)
plt.scatter(xi, yi, s=1)

# xf, yf, xi, yi = get_two_timepoints_two_vars(df=df28,
#                                              col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
#                                              col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
#                                              ti=0, tf=19)

# plt.scatter(xf, yf, s=3)
# plt.scatter(xi, yi, s=3)

_, df_sub = get_subset_df(df=df26, GO_id=trna_synthetases_GO)
xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                      col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                      col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
                                      ti=None, tf=19)
plt.scatter(xf, yf, s=5)

# _, df_sub = get_subset_df(df=df28, GO_id=trna_synthetases_GO)
# xf, yf  = get_two_timepoints_two_vars(df=df_sub,
#                                       col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
#                                       col_y='Kernel Trace: major_axis_length',
#                                       ti=None, tf=19)
# plt.scatter(xf, yf, s=5)